In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import os, time

In [ ]:
#-----------------------------------------------------------------------------#
#    IMAGE PREPROCESSING                                                      #
#-----------------------------------------------------------------------------#
## DIRECTORIES with datasets
img_path = 'dataset'

# Dataset with Transformation
dataset = datasets.ImageFolder(
    img_path,
    transforms.Compose([
        transforms.Resize((128, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
              mean=[0.0, 0.0, 0.0], 
              std=[1.0, 1.0, 1.0])])
)

# Data split
trainpctg  = 0.8
train_size = int(trainpctg * len(dataset))
test_size = len(dataset) - train_size
trainset, testset = random_split(dataset, [train_size, test_size])

print(f"Training set size: {len(trainset)}, Testing set size: {len(testset)}")

In [ ]:
#-----------------------------------------------------------------------------#
#    DATA LOADER                                                              #
#-----------------------------------------------------------------------------#
# For a smaller subset
smallsz     = 50
subidx      = np.random.choice(len(trainset), smallsz, replace=False)
trainsubset = torch.utils.data.Subset(trainset, subidx)
subidx      = np.random.choice(len(testset),  smallsz, replace=False)
testsubset  = torch.utils.data.Subset(testset, subidx)

large = True
batchsz  = 64
sys_workers = int(os.getenv("SLURM_CPUS_PER_TASK", 4))  # Default to 4 if not set


# Data Loaderd
if large:
    trainloader = DataLoader(trainset, 
                             batch_size=batchsz, 
                             shuffle=True, 
                             drop_last=True, 
                             num_workers=sys_workers, 
                             pin_memory=True, 
                             prefetch_factor=2)
    testloader = DataLoader(testset,
                            batch_size=batchsz,
                            shuffle=False,
                            drop_last=False,
                            num_workers=sys_workers,
                            pin_memory=True)  # Optional but recommended if using a GPU
else:
    trainloader = DataLoader(trainsubset, batch_size=1, shuffle=True, num_workers=sys_workers)
    testloader = DataLoader(testsubset , batch_size=1, shuffle=False, num_workers=sys_workers)

# print(dataset)
# print(dataloader)
print("Min/Max:", dataset[0][0].min(), dataset[0][0].max())

# Test CPU

In [ ]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = vaemodel1()

In [ ]:
model.load_state_dict(torch.load("pre_trained_w_encoder.pt", weights_only=True))

In [ ]:
model.eval()
with torch.no_grad():
    start_time = time.time()
    cpu_z = []
    for images, _ in testloader:
        # Run inference for each image individually
        for i in range(images.shape[0]):
            image_single = images[i:i+1]  # shape (1, C, H, W)
            pred = model(image_single)
            cpu_z.append(pred)

    end_time = time.time()
    total_time = end_time - start_time
    # Concatenate all the batch outputs to form a predictions vector
    predictions_vector = torch.cat(cpu_z, dim=0)

    # calculate average inference time and FPS
    num_images = len(predictions_vector)
    avg_inference_time = total_time / num_images
    fps = num_images / total_time

    print("Total inference time: {:.6f} seconds".format(total_time))
    print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
    print("FPS: {:.2f}".format(fps))

# Test DPU

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")

In [ ]:
overlay.load_model("zcu104_vaemodel1.xmodel")

In [ ]:
dpu = overlay.runner

inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn = tuple(inputTensors[0].dims)
shapeOut = tuple(outputTensors[0].dims)
outputSize = int(outputTensors[0].get_data_size() / shapeIn[0])

In [ ]:
output_data = [np.empty(shapeOut, dtype=np.float32, order="C")]
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]
data = input_data[0]

In [ ]:
inference_time = 0
vitisAI_z = []
for images, _ in testloader:
    for i in range(images.shape[0]):
        data[0] = images[i:i+1]
        time1 = time.time()
        job_id = dpu.execute_async(input_data, output_data)
        dpu.wait(job_id)
        o_data = output_data[0].reshape(2, 64)
        std = np.exp(o_data[1]*0.5)
        eps = np.random.randn(*std.shape)
        vitisAI_z.append(o_data[0]+eps*std)
        time2 = time.time()
        inference_time += time2-time1
num_images = len(vitisAI_z)
avg_infer_time = inference_time/num_images
fps = num_images/inference_time
print("Average inference time: {} s".format(avg_infer_time))
print("Performance: {} FPS".format(fps))

# Test FINN